<a href="https://colab.research.google.com/github/CPernet/Semantically_Incongruent_or_Congruent_Eggplants_revised/blob/main/erps_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Trial-level ERP analysis**

This notebook performs the ERP-analysis stage of the revised trial-level reanalysis of the Toffolo et al. (2022) N400 dataset.

This notebook links the predictors from stimuli_analysis.ipynb to individual EEG trials and performs both participant-level and group-level statistical analyses.

The pipeline first exports the retained ERP data into a long-format representation, preserving the correspondence between each retained EEG epoch and its original experimental event. Participant-specific design matrices are then constructed by matching every retained trial with its linguistic predictors. These design matrices are subsequently used to estimate participant-level mass-univariate general linear models across electrodes and time points. Finally, component-level linear mixed-effects models are fitted to the Recognition Potential (RP), N400 and Late Positive Component (LPC) in order to quantify the relationship between linguistic predictors and ERP amplitudes while accounting for repeated observations across participants and stimuli.

**Export retained ERP trials**

This section converts the processed ERP derivatives into a trial-level long-format dataset suitable for statistical analysis.

Each retained EEG epoch is matched to its original experimental event using the EEGLAB `urevent` information and target-word onset. The corresponding stimulus identifier (`stim_key`) is recovered and combined with the EEG amplitudes, electrode metadata and temporal information.

The resulting dataset preserves every retained EEG observation and provides the common input for the subsequent ERP analyses. Participant-specific retained-trial lookup tables are also generated for later construction of the first-level design matrices.

In [9]:
%cd /content

!rm -rf Semantically_Incongruent_or_Congruent_Eggplants_revised
!git clone https://github.com/CPernet/Semantically_Incongruent_or_Congruent_Eggplants_revised.git

%cd /content/Semantically_Incongruent_or_Congruent_Eggplants_revised

/content
Cloning into 'Semantically_Incongruent_or_Congruent_Eggplants_revised'...
remote: Enumerating objects: 394, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 394 (delta 76), reused 48 (delta 29), pack-reused 280 (from 1)
Receiving objects: 100% (394/394), 94.77 MiB | 47.43 MiB/s, done.
Resolving deltas: 100% (224/224), done.
/content/Semantically_Incongruent_or_Congruent_Eggplants_revised


In [10]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 95.6 MB/s eta 0:00:00


In [11]:
!unzip language_outputs.zip

Archive:  language_outputs.zip
   creating: language_outputs/
  inflating: language_outputs/ALL_language_metrics.tsv  
  inflating: language_outputs/ALL_predictor_diagnostics_vif.tsv  
  inflating: language_outputs/ALL_predictor_diagnostics_correlations.tsv  
  inflating: language_outputs/ALL_language_metrics_GLM.tsv  


**Note**: Due to their size, the N400 ERP derivatives are stored on Google Drive and mounted during notebook execution.

In [12]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!unzip -o erps.zip

Archive:  erps.zip
  inflating: erps/task-N400Stimset_erp-CP_trialrej.json  
  inflating: erps/task-N400Stimset_erp-GA_filter.json  
  inflating: erps/task-N400Stimset_erp-GA_trialrej.json  
  inflating: erps/task-N400Stimset_erp-LD_trialrej.json  
  inflating: erps/task-N400Stimset_erp-Order_trialrej.json  
  inflating: erps/task-N400Stimset_erp-Time_trialrej.json  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-CP.mat  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-CP_trialrej.tsv  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-GA.mat  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-GA_trialrej.tsv  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-LD.mat  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-LD_trialrej.tsv  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-Order.mat  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-Order_trialrej.tsv  
  inflating: erps/sub-01/sub-01_task-N400Stimset_erp-Time.mat  
  inflating: erps/sub-01/sub

In [13]:
%cd /content/Semantically_Incongruent_or_Congruent_Eggplants_revised/

/content/Semantically_Incongruent_or_Congruent_Eggplants_revised


# **ERP export analyses**

The ERP derivatives are already split into five analysis schemes: CP, GA, LD, Order, and Time. Each analysis has its own `.mat` file and matching `*_trialrej.tsv` file for each subject. The export script reads those existing files and exports the retained ERP trials to TSV.

Each output keeps the subject, analysis name, condition number, condition label, retained-trial index, original EEGLAB `urevent_index`, channel, timepoint, and amplitude. If the retained-trial count in the `.mat` file differs from the count reported in the `*_trialrej.tsv`, the script uses the `.mat` count because the `.mat` file contains the actual ERP data.

### CP

Exports the Cloze Probability analysis. Trials are grouped by congruency and predictability band, so this output keeps whether each sentence ending was congruent/incongruent and how predictable it was.

In [ ]:
!python erp_analysis/export_erp_long.py \
  "/content/drive/MyDrive/N400/erps" \
  --analyses CP \
  --output-dir "/content/drive/MyDrive/N400/eeg_outputs_CP"

Found 20 ERP MAT files.
Using ERP root: /content/drive/MyDrive/N400/erps
Analyses: CP
Channels: ALL

Processing sub-01_task-N400Stimset_erp-CP.mat
  Using sub-01_task-N400Stimset_erp-CP_trialrej.tsv
  Channel filter: ALL channels
  Condition 1: Congruent: Cloze Probability ≥ 96 % - 54 before rejection, 29 reported retained, 29 MAT retained, 615 timepoints, 128 of 128 channels exported
  Condition 2: Congruent: 96 % > Cloze Probability ≥ 90 % - 56 before rejection, 40 reported retained, 40 MAT retained, 615 timepoints, 128 of 128 channels exported
  Condition 3: Congruent: 90 % > Cloze Probability ≥ 80 % - 47 before rejection, 30 reported retained, 30 MAT retained, 615 timepoints, 128 of 128 channels exported
  Condition 4: Congruent: Cloze Probability < 80 % - 43 before rejection, 26 reported retained, 26 MAT retained, 615 timepoints, 128 of 128 channels exported
  Condition 5: Incongruent: Cloze Probability ≥ 96 % - 55 before rejection, 33 reported retained, 33 MAT retained, 615 timep

In [ ]:
!python erp_analysis/export_erp_long.py \
  "/content/drive/MyDrive/N400/erps" \
  --analyses GA \
  --output-dir "/content/drive/MyDrive/N400/eeg_outputs_GA"

In [ ]:
!python erp_analysis/export_erp_long.py \
  "/content/drive/MyDrive/N400/erps" \
  --analyses LD \
  --output-dir "/content/drive/MyDrive/N400/eeg_outputs_LD"

In [ ]:
!python erp_analysis/export_erp_long.py \
  "/content/drive/MyDrive/N400/erps" \
  --analyses Order \
  --output-dir "/content/drive/MyDrive/N400/eeg_outputs_Order"

In [ ]:
!python erp_analysis/export_erp_long.py \
  "/content/drive/MyDrive/N400/erps" \
  --analyses Time \
  --output-dir "/content/drive/MyDrive/N400/eeg_outputs_Time"